# InfraVision: train the person detector on POP (disaster-area drone thermal images)

Fine-tunes the thermal YOLO11s model you trained on FLIR (`thermal_best.pt`) on the POP dataset:
top-down drone views at 30 / 50 / 70 m, 1280×1024 images, people often only 20–40 px tall.

**Why `imgsz=1024`:** at 640 px about a third of the people shrink below 16 px and become very hard to detect.

**Before running:**
1. Upload `pop_yolo_colab.zip` to Google Drive → `My Drive/Infravision/pop_yolo_colab.zip`.
2. **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**.

Progress is saved to Drive after every epoch. Free Colab sessions can end after a few hours:
if that happens, reconnect and **Run all** again — it continues from the last saved epoch.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Use Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU ready:", torch.cuda.get_device_name(0))

## 2. Install the same Ultralytics version used on the laptop

In [ ]:
!pip install -q ultralytics==8.4.177
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. Settings, Google Drive and paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/Infravision")
ZIP_PATH = DRIVE_DIR / "pop_yolo_colab.zip"
RUNS_DIR = DRIVE_DIR / "runs"
RUN_NAME = "pop_1024"
RUN_DIR = RUNS_DIR / RUN_NAME
DATA_YAML = Path("/content/pop/data.yaml")
START_WEIGHTS = Path("/content/weights/thermal_best.pt")

IMGSZ = 1024     # keep small people visible
EPOCHS = 60
PATIENCE = 15    # stop early if val mAP does not improve for 15 epochs
BATCH = 8        # fits a T4 (16 GB) at 1024 px

assert ZIP_PATH.exists(), f"Zip not found: {ZIP_PATH}. Upload it to My Drive/Infravision/ first."
print("Zip :", ZIP_PATH)
print("Run :", RUN_DIR)

## 4. Unpack the dataset onto the Colab disk

In [ ]:
import shutil, zipfile

if not DATA_YAML.exists():
    local_zip = Path("/content/pop_yolo_colab.zip")
    shutil.copy(ZIP_PATH, local_zip)          # local disk is much faster than reading from Drive
    with zipfile.ZipFile(local_zip) as z:
        z.extractall("/content")
    local_zip.unlink()

for split in ("train", "val", "test"):
    n_img = len(list(Path(f"/content/pop/images/{split}").iterdir()))
    n_lbl = len(list(Path(f"/content/pop/labels/{split}").iterdir()))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")
print(DATA_YAML.read_text())

## 5. Baseline: the FLIR model on POP *before* training

Gives the "before" number for your report.

In [ ]:
from ultralytics import YOLO

before = YOLO(str(START_WEIGHTS)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                                      device=0, plots=False, project="/content/val", name="before")
print(f"BEFORE  precision={before.box.mp:.3f} recall={before.box.mr:.3f} "
      f"mAP50={before.box.map50:.3f} mAP50-95={before.box.map:.3f}")

## 6. Train on POP

Starts from the FLIR thermal model, or continues from Drive if a previous session was interrupted.
Roughly 4–5 minutes per epoch on a T4 at 1024 px.

In [ ]:
drive_last = RUN_DIR / "weights" / "last.pt"

if drive_last.exists():
    print("Continuing interrupted run from:", drive_last)
    try:
        YOLO(str(drive_last)).train(resume=True, data=str(DATA_YAML), device=0, workers=2)
    except AssertionError as e:
        print("Nothing to resume (training already finished):", e)
else:
    print("Starting new run from:", START_WEIGHTS)
    YOLO(str(START_WEIGHTS)).train(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=BATCH,
        device=0,
        workers=2,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        # Drone images are top-down, so vertical flips are realistic too
        flipud=0.5,
    )

## 7. Evaluate on the POP test split (never used for training)

In [ ]:
best = RUN_DIR / "weights" / "best.pt"
after = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                            device=0, project="/content/val", name="after")

print()
print(f"{'':8s} {'precision':>9s} {'recall':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
print(f"{'BEFORE':8s} {before.box.mp:9.3f} {before.box.mr:7.3f} {before.box.map50:7.3f} {before.box.map:9.3f}")
print(f"{'AFTER':8s} {after.box.mp:9.3f} {after.box.mr:7.3f} {after.box.map50:7.3f} {after.box.map:9.3f}")

## 8. Training curve and example detections

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display

df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()
display(df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10))
display(Show(str(RUN_DIR / "results.png"), width=900))

preds = YOLO(str(best)).predict(sorted(Path("/content/pop/images/test").iterdir())[::150][:6],
                                imgsz=IMGSZ, conf=0.25, iou=0.5, save=True,
                                project="/content/pred", name="pop", exist_ok=True)
for p in sorted(Path("/content/pred/pop").iterdir()):
    display(Show(str(p), width=640))

## 9. Download the results

Downloads `pop_1024.zip` (weights, `results.csv`, plots). On the laptop, unzip it into
`human_detection/runs/pop/pop_1024/`. The files also stay in `My Drive/Infravision/runs/pop_1024`.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/pop_1024", "zip", RUN_DIR)
files.download(archive)